# Nino3.4 and PDO
Blob-analog identification (canonical + non-canonical/actual footprint) and per-analog Nino3.4/PDO climate state, across all four SSTa detrending methods.

### Imports

In [1]:
import json
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

### Data

In [2]:
BASE_PATHS = {
    'linear':    '/glade/work/cmendiola/data_conv_lin_trend',
    'quadratic': '/glade/work/cmendiola/data_quad_trend',
    'ensmean':   '/glade/work/cmendiola/data_ens_mean',
    'noseas':    '/glade/work/cmendiola/data_rm_seasonalcycle_mean',
}

SSTA_VARNAMES = {
    'linear': '__xarray_dataarray_variable__',
    'quadratic': '__xarray_dataarray_variable__',
    'ensmean': 'SST',
    'noseas': '__xarray_dataarray_variable__',
}

OBJECT_ID_JSON_PATHS = {
    'linear':    'object_id_ls_greenmask_linear_corrected.json',
    'quadratic': 'object_id_ls_greenmask_quadratic_corrected.json',
    'ensmean':   'object_id_ls_greenmask_ensmean_corrected.json',
    'noseas':    'object_id_ls_greenmask_noseas_corrected.json',
}

RADIUS_INDEX_FOR_2DEG = 1  # index 1 = 2-degree structuring element, confirmed across all four methods
OVERLAP_THRESHOLD_INDEX = 4  # index 4 of thresholds = 0.5 (50% overlap)
THRESHOLDS = np.arange(0.1, 0.81, 0.1)
TIME_SLICE = slice('1979-01', '2020-12')

NINO34_PATH = '/glade/derecho/scratch/cassiacai/nino34_lens_full_basin.nc'
PDO_PATH = '/glade/derecho/scratch/cassiacai/pdo_lens_full_basin.nc'
CANONICAL_MASK_PATH = 'mean_mask_3.nc'

### Functions

In [3]:
def load_mhwobj_and_ssta(method, base_path, varname):
    """Load MHW object labels (radius-corrected, time-restricted) and SSTa for one method."""
    mhwobj_paths = [f'{base_path}/ens_{i}_mhwobj.nc' for i in range(100)]
    ssta_paths = [f'{base_path}/ens_{i}_ssta.nc' for i in range(100)]

    labels = (
        xr.open_mfdataset(mhwobj_paths, combine='nested', concat_dim='ensemble_member')
        .labels.sel(radius=RADIUS_INDEX_FOR_2DEG)
        .sel(time=TIME_SLICE)
        .compute()
    )
    ssta = xr.open_mfdataset(ssta_paths, combine='nested', concat_dim='ensemble_member').compute()
    return labels, ssta

In [4]:
def load_all_methods():
    """Load labels + SSTa for all four detrending methods."""
    labels_by_method, ssta_by_method = {}, {}
    for method, base_path in BASE_PATHS.items():
        labels, ssta = load_mhwobj_and_ssta(method, base_path, SSTA_VARNAMES[method])
        labels_by_method[method] = labels
        ssta_by_method[method] = ssta
        print(f"[{method}] loaded: {labels.sizes}")
    return labels_by_method, ssta_by_method

In [5]:
def load_blob_analog_ids():
    """Load the corrected Blob-analog object IDs at the 50% overlap threshold, per method."""
    analog_ids = {}
    for method, path in OBJECT_ID_JSON_PATHS.items():
        with open(path, 'r') as f:
            all_thresholds = json.load(f)
        analog_ids[method] = all_thresholds[OVERLAP_THRESHOLD_INDEX]
    return analog_ids

In [6]:
def load_canonical_mask():
    """Load and binarize the canonical NE Pacific footprint mask."""
    mask = xr.open_dataset(CANONICAL_MASK_PATH).mhw_obj
    return xr.where(mask > 0.1, 1, 0)

In [7]:
def load_climate_indices():
    """Load Nino3.4 and PDO index datasets, confirming variable names."""
    nino_ds = xr.open_dataset(NINO34_PATH)
    pdo_ds = xr.open_dataset(PDO_PATH)
    print("Nino3.4 vars:", list(nino_ds.data_vars))
    print("PDO vars:", list(pdo_ds.data_vars))

    nino_var = list(nino_ds.data_vars)[0]
    pdo_var = list(pdo_ds.data_vars)[0]
    return nino_ds[nino_var], pdo_ds[pdo_var]

In [8]:
def compute_cell_area(grid_ref, R=6371.0):
    """Area-weighting for the CESM2-LENS grid (km^2 per gridcell, function of latitude)."""
    dlat = np.deg2rad(np.mean(np.diff(grid_ref.lat.values)))
    dlon = np.deg2rad(np.mean(np.diff(grid_ref.lon.values)))
    cell_area = (R ** 2) * dlat * dlon * np.cos(np.deg2rad(grid_ref.lat))
    return xr.DataArray(cell_area.values, coords={'lat': grid_ref.lat}, dims='lat')

In [9]:
def calc_frac_overlap(one_obj, mask):
    """Max fractional overlap (over the object's full lifetime) with a fixed reference mask."""
    one_obj_binary = xr.where(one_obj > 0, 1., 0)
    overlap_only = xr.where((one_obj_binary + mask) > 1, 1., 0)
    overlap_frac = overlap_only.sum(dim=('lat', 'lon')) / mask.sum()
    if overlap_frac.size > 0 and overlap_frac.notnull().any():
        return overlap_frac.max().item()
    return 0.0

In [10]:
def compute_climate_stats(labels_by_method, blob_analog_ids, nino_da, pdo_da):
    """
    For every Blob-analog (across all four detrending methods), compute mean/max
    Nino3.4 and PDO over the analog's active months.
    """
    climate_stats = {}
    for method, labels_full in labels_by_method.items():
        method_ids = blob_analog_ids[method]
        for member_id in range(100):
            member_labels = labels_full.isel(ensemble_member=member_id)
            member_nino = nino_da.sel(member=member_id)
            member_pdo = pdo_da.sel(member=member_id)

            for analog_id in method_ids[member_id]:
                mask = (member_labels == analog_id)
                times_present = member_labels.time.where(mask.any(dim=('lat', 'lon')), drop=True)
                if len(times_present) == 0:
                    continue

                nino_vals = member_nino.sel(time=times_present)
                pdo_vals = member_pdo.sel(time=times_present)

                climate_stats[(method, member_id, analog_id)] = {
                    'nino34_mean': float(nino_vals.mean()),
                    'nino34_max': float(nino_vals.max()),
                    'pdo_mean': float(pdo_vals.mean()),
                    'pdo_max': float(pdo_vals.max()),
                }
        print(f"[{method}] climate context computed for "
              f"{sum(1 for k in climate_stats if k[0] == method)} analogs")

    return climate_stats

In [11]:
def load_fosi_blob(object_id=94.0, expected_n_months=20, ssta_varname=None):
    """Load the FOSI Blob's own detection window and sanity-check against known values."""
    fosi_blobs = xr.open_dataset('fosi_blobs_r2.nc').labels
    blob_times = fosi_blobs.time.where((fosi_blobs == object_id).any(dim=('lat', 'lon')), drop=True)

    regridded_ssta_ds = xr.open_dataset('/glade/derecho/scratch/cassiacai/regridded_ssta_no_trend_NEP.nc')

    # BUG FIX: xr.open_dataset() returns a Dataset. Dataset.values is NOT the
    # underlying array -- it's the inherited dict-like Mapping.values() *method*,
    # since Dataset implements the Mapping interface. Calling float() on that
    # bound method raises "not 'method'". Must select the actual DataArray
    # variable first (Dataset[varname]), which DOES expose a real .values ndarray.
    if ssta_varname is None:
        ssta_varname = list(regridded_ssta_ds.data_vars)[0]
        print(f"(inferred SSTa variable name: '{ssta_varname}')")
    regridded_ssta = regridded_ssta_ds[ssta_varname]

    max_val_check = regridded_ssta.sel(time='2015-07').max().values

    print(f"FOSI Blob sanity check: max SSTa Jul 2015 = {float(max_val_check):.2f} (expect ~2.34)")
    print(f"n_months in blob_times = {len(blob_times)} (expect {expected_n_months})")

    return fosi_blobs, blob_times

In [12]:
def build_fosi_blob_footprint_mask(fosi_blobs, object_id=94.0):
    """
    Build a reference mask from the FOSI Blob's OWN tracked footprint (union of every
    gridpoint the object ever occupied across its full lifetime), as an alternative
    reference region to the independently-derived canonical cluster mask (mask_3).

    This is the "actual footprint" reference: rather than asking whether a LENS
    candidate overlaps a separately-computed climatological NE Pacific region, this
    asks whether it overlaps the literal shape the Blob itself traced out in FOSI.
    """
    is_blob = (fosi_blobs == object_id)
    footprint_union = is_blob.any(dim='time')  # True at any gridpoint ever occupied
    return xr.where(footprint_union, 1, 0)

In [13]:
def select_analogs_noncanonical(labels_by_method, fosi_footprint_mask, threshold=0.5):
    """
    Select Blob-analogs using the FOSI Blob's actual tracked footprint as the
    reference region (non-canonical), applying the same max-lifetime-overlap
    logic as calc_frac_overlap, at the same 50% threshold used for the
    canonical selection.

    Returns: dict[method] -> dict[member_id] -> list[analog_id]
    """
    analog_ids_noncanonical = {}
    for method, labels_full in labels_by_method.items():
        method_ids = {}
        for member_id in range(labels_full.sizes['ensemble_member']):
            member_labels = labels_full.isel(ensemble_member=member_id)
            candidate_ids = np.unique(member_labels.values)
            candidate_ids = candidate_ids[~np.isnan(candidate_ids)]
            candidate_ids = candidate_ids[candidate_ids > 0]  # drop background/0 label

            matched = []
            for cand_id in candidate_ids:
                one_obj = member_labels.where(member_labels == cand_id, 0)
                frac = calc_frac_overlap(one_obj, fosi_footprint_mask)
                if frac >= threshold:
                    matched.append(cand_id)
            method_ids[member_id] = matched
        analog_ids_noncanonical[method] = method_ids
        n_total = sum(len(v) for v in method_ids.values())
        print(f"[{method}] non-canonical (actual-footprint) analogs: {n_total}")
    return analog_ids_noncanonical

In [14]:
def plot_reference_masks(canonical_mask, fosi_footprint_mask, savepath='reference_masks_comparison.png'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    canonical_mask.plot(ax=axes[0], add_colorbar=False, cmap='Blues')
    axes[0].set_title('Canonical NE Pacific footprint\n(cluster-derived, mask_3)')

    fosi_footprint_mask.plot(ax=axes[1], add_colorbar=False, cmap='Reds')
    axes[1].set_title("FOSI Blob's own tracked footprint\n(actual, full-lifetime union)")

    for ax in axes:
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

    plt.tight_layout()
    plt.savefig(savepath, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved reference mask comparison to {savepath}")

In [15]:
def select_analogs_noncanonical_with_candidates(labels_by_method, fosi_footprint_mask, threshold=0.5):
    """
    Same selection logic as select_analogs_noncanonical, but also records
    EVERY candidate object considered (with its overlap fraction), not just
    the ones that passed the threshold. Useful for auditing/debugging and
    for direct comparison against the canonical selection's candidate pool.

    Returns:
        analog_ids_noncanonical: dict[method] -> dict[member_id] -> list[matched analog_id]
        all_candidates: dict[method] -> dict[member_id] -> dict[candidate_id -> overlap_frac]
    """
    analog_ids_noncanonical = {}
    all_candidates = {}

    for method, labels_full in labels_by_method.items():
        method_ids = {}
        method_candidates = {}
        for member_id in range(labels_full.sizes['ensemble_member']):
            member_labels = labels_full.isel(ensemble_member=member_id)
            candidate_ids = np.unique(member_labels.values)
            candidate_ids = candidate_ids[~np.isnan(candidate_ids)]
            candidate_ids = candidate_ids[candidate_ids > 0]

            matched = []
            candidate_overlaps = {}
            for cand_id in candidate_ids:
                one_obj = member_labels.where(member_labels == cand_id, 0)
                frac = calc_frac_overlap(one_obj, fosi_footprint_mask)
                candidate_overlaps[float(cand_id)] = frac
                if frac >= threshold:
                    matched.append(cand_id)

            method_ids[member_id] = matched
            method_candidates[member_id] = candidate_overlaps
        analog_ids_noncanonical[method] = method_ids
        all_candidates[method] = method_candidates
        n_total = sum(len(v) for v in method_ids.values())
        n_candidates_total = sum(len(v) for v in method_candidates.values())
        print(f"[{method}] non-canonical matches: {n_total} / {n_candidates_total} candidates considered")

    return analog_ids_noncanonical, all_candidates

In [16]:
def plot_reference_masks_shared_extent(mask_a, mask_b, title_a, title_b,
                                        lat_range=(0, 60), lon_range=(150, 250),
                                        savepath='reference_masks_comparison.png'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    mask_a.plot(ax=axes[0], add_colorbar=False, cmap='Blues')
    mask_b.plot(ax=axes[1], add_colorbar=False, cmap='Reds')
    axes[0].set_title(title_a)
    axes[1].set_title(title_b)
    for ax in axes:
        ax.set_xlim(lon_range)
        ax.set_ylim(lat_range)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(savepath, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved to {savepath}")

In [22]:
def unflatten_climate_stats(flat_dict):
    """Convert 'method_memberid_analogid' string keys back to (method, member_id, analog_id) tuples."""
    unflattened = {}
    for flat_key, v in flat_dict.items():
        parts = flat_key.split('_')
        method = parts[0]
        member_id = int(parts[1])
        analog_id = float(parts[2])
        unflattened[(method, member_id, analog_id)] = v
    return unflattened

In [23]:
def load_climate_stats_if_exists(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            flat = json.load(f)
        stats = unflatten_climate_stats(flat)
        print(f"Loaded {path}: {len(stats)} total entries across all methods")
        return stats
    else:
        print(f"NOT FOUND: {path}")
        return None

In [24]:
def build_fosi_blob_footprint_mask_peak_month(fosi_blobs, object_id=94.0):
    """
    Build a reference mask from the FOSI Blob's footprint at its single month
    of PEAK SPATIAL EXTENT (per manuscript: September 2015), rather than the
    full-lifetime union. This gives a reference region much closer in size to
    the canonical mask, avoiding the 2.11x inflation caused by unioning 20
    months of a migrating, growing event.
    """
    is_blob = (fosi_blobs == object_id)
    n_gridpoints_per_month = is_blob.sum(dim=('lat', 'lon'))
    peak_idx = int(n_gridpoints_per_month.argmax(dim='time'))
    peak_month_time = n_gridpoints_per_month.time.isel(time=peak_idx)

    footprint_at_peak = is_blob.sel(time=peak_month_time)
    mask = xr.where(footprint_at_peak, 1, 0)
    return mask, peak_month_time

## Analysis

In [ ]:
labels_by_method, ssta_by_method = load_all_methods()canonical_mask = load_canonical_mask()
nino_da, pdo_da = load_climate_indices()
cell_area_da = compute_cell_area(labels_by_method['linear'])
fosi_blobs, fosi_blob_times = load_fosi_blob()

canonical_area_km2 = float((canonical_mask * cell_area_da).sum())
print(f"Canonical mask area: {canonical_area_km2:,.0f} km^2")

In [25]:
climate_stats_canonical = load_climate_stats_if_exists(
    '/glade/derecho/scratch/cassiacai/climate_stats_canonical.json'
)
climate_stats_noncanonical = load_climate_stats_if_exists(
    '/glade/derecho/scratch/cassiacai/climate_stats_noncanonical.json'
)

Loaded /glade/derecho/scratch/cassiacai/climate_stats_canonical.json: 1313 total entries across all methods
Loaded /glade/derecho/scratch/cassiacai/climate_stats_noncanonical.json: 504 total entries across all methods


In [3]:
# Population 1: canonical footprint (fixed cluster-derived mask_3) ---
climate_stats_canonical = compute_climate_stats(
        labels_by_method, blob_analog_ids_canonical, nino_da, pdo_da)
with open('/glade/derecho/scratch/cassiacai/climate_stats_canonical.json', 'w') as f:
    json.dump({f"{k[0]}_{k[1]}_{k[2]}": v for k, v in climate_stats_canonical.items()}, f, indent=2)
print(f"\n[canonical] Total analogs with climate context: {len(climate_stats_canonical)}")

# Population 2: non-canonical, "actual footprint" (FOSI Blob's own tracked shape) ---
fosi_footprint_mask = build_fosi_blob_footprint_mask(fosi_blobs, object_id=94.0)
blob_analog_ids_noncanonical = select_analogs_noncanonical(labels_by_method, fosi_footprint_mask)
climate_stats_noncanonical = compute_climate_stats(
    labels_by_method, blob_analog_ids_noncanonical, nino_da, pdo_da
)

with open('/glade/derecho/scratch/cassiacai/climate_stats_noncanonical.json', 'w') as f:
    json.dump({f"{k[0]}_{k[1]}_{k[2]}": v for k, v in climate_stats_noncanonical.items()}, f, indent=2)
print(f"[non-canonical] Total analogs with climate context: {len(climate_stats_noncanonical)}")

n_canon_linear = sum(1 for k in climate_stats_canonical if k[0] == 'linear')
n_noncanon_linear = sum(1 for k in climate_stats_noncanonical if k[0] == 'linear')
print(f"\n[linear method] canonical n={n_canon_linear} vs. non-canonical (actual footprint) n={n_noncanon_linear}")

[linear] loaded: Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})
[quadratic] loaded: Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})
[ensmean] loaded: Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})
[noseas] loaded: Frozen({'ensemble_member': 100, 'time': 504, 'lat': 64, 'lon': 81})
Nino3.4 vars: ['__xarray_dataarray_variable__']
PDO vars: ['__xarray_dataarray_variable__']
Equatorial gridcell area check: 11881.4 km^2
(inferred SSTa variable name: '__xarray_dataarray_variable__')
FOSI Blob sanity check: max SSTa Jul 2015 = 2.34 (expect ~2.34)
n_months in blob_times = 20 (expect 20)
[linear] climate context computed for 331 analogs
[quadratic] climate context computed for 327 analogs
[ensmean] climate context computed for 312 analogs
[noseas] climate context computed for 343 analogs

[canonical] Total analogs with climate context: 1313
[linear] non-canonical (actual-footprint) analogs: 118
[quadratic] non-canonical (actual-footp

In [26]:
if climate_stats_canonical is not None:
    for method in labels_by_method:
        n = sum(1 for k in climate_stats_canonical if k[0] == method)
        print(f"[canonical/{method}] {n} analogs")

if climate_stats_noncanonical is not None:
    for method in labels_by_method:
        n = sum(1 for k in climate_stats_noncanonical if k[0] == method)
        print(f"[non-canonical full/{method}] {n} analogs")

[canonical/linear] 331 analogs
[canonical/quadratic] 327 analogs
[canonical/ensmean] 312 analogs
[canonical/noseas] 343 analogs
[non-canonical full/linear] 118 analogs
[non-canonical full/quadratic] 106 analogs
[non-canonical full/ensmean] 103 analogs
[non-canonical full/noseas] 177 analogs


In [38]:
blob_climate = {
    'nino34_mean': 1.1281033521296626,
    'nino34_max': 3.0774558632413305,
    'pdo_mean': 1.0245222060799457,
    'pdo_max': 1.8462614767620456,
}

with open('/glade/derecho/scratch/cassiacai/blob_climate.json', 'w') as f:
    json.dump(blob_climate, f, indent=2)

print("Saved blob_climate to /glade/derecho/scratch/cassiacai/blob_climate.json")
print(blob_climate)

Saved blob_climate to /glade/derecho/scratch/cassiacai/blob_climate.json
{'nino34_mean': 1.1281033521296626, 'nino34_max': 3.0774558632413305, 'pdo_mean': 1.0245222060799457, 'pdo_max': 1.8462614767620456}
